# Qwen3-32B


In [ ]:
# Cell 1: Setup (Qwen planner pipeline via OpenRouter, independent)
import os
import re
import json
import time
import subprocess
import requests
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv

# Project root (run notebook from formalizer-scaling/src/planner_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Data / prompt / output paths
DATA_DIR = PROJECT_ROOT / "data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "llm_planner_instruction.txt"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"

# New output root for direct LLM-as-planner pipeline
MODEL_DIRNAME = "qwen3_32b"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "llm_planner" / MODEL_DIRNAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# VAL executable (adjust if your environment differs)
VAL_PATH = Path.home() / "tools" / "VAL" / "build" / "linux64" / "Release" / "bin" / "Validate"

# Model + API
MODEL_NAME = "qwen/qwen3-32b"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
  raise ValueError(f"OPENROUTER_API_KEY not found in {ENV_PATH}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

In [ ]:
# Cell 2: Load problem IDs, planner instruction, and ground-truth domain PDDL
import pandas as pd

# Use all available problems by default; edit this list later for targeted runs
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 201)]

PLANNER_INSTRUCTION = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Planner prompt chars:", len(PLANNER_INSTRUCTION))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

In [ ]:
# Cell 3: Helper - load NL domain/problem descriptions for one problem ID
def load_descriptions(problem_id: str):
  domain_desc_path = DATA_DIR / "descriptions" / f"{problem_id}_domain.txt"
  problem_desc_path = DATA_DIR / "descriptions" / f"{problem_id}_problem.txt"

  if not domain_desc_path.exists():
      raise FileNotFoundError(f"Missing domain description: {domain_desc_path}")
  if not problem_desc_path.exists():
      raise FileNotFoundError(f"Missing problem description: {problem_desc_path}")

  return domain_desc_path.read_text(), problem_desc_path.read_text()

In [ ]:
# Cell 4: Helper - build planner prompt (NL domain + NL problem + GT domain PDDL)
def build_prompt(problem_id: str):
  domain_desc, problem_desc = load_descriptions(problem_id)

  return (
      PLANNER_INSTRUCTION
      + "\n\nYou are an expert automated planner.\n"
      "Create an executable plan that is consistent with all provided information.\n\n"
      "### Natural-language domain description\n"
      f"{domain_desc}\n\n"
      "### Natural-language problem description\n"
      f"{problem_desc}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output only the final plan wrapped in <plan>...</plan>."
  )

In [ ]:
# Cell 5: Call Qwen3-32B via OpenRouter
def call_qwen(prompt: str) -> str:
  api_key = os.getenv("OPENROUTER_API_KEY")
  if not api_key:
      raise RuntimeError("OPENROUTER_API_KEY is not set in environment.")

  headers = {
      "Authorization": f"Bearer {api_key}",
      "Content-Type": "application/json",
  }

  payload = {
      "model": MODEL_NAME,
      "messages": [{"role": "user", "content": prompt}],
      "temperature": 0.4,
      "max_tokens": 32768,
  }

  resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=300)
  if not resp.ok:
      raise RuntimeError(
          f"OpenRouter request failed: {resp.status_code} {resp.reason}\n{resp.text}"
      )

  data = resp.json()
  return data["choices"][0]["message"]["content"] or ""

In [ ]:
# Cell 6: Helper - extract <plan>...</plan> block
def extract_plan(raw_text: str):
  if not raw_text:
      return ""

  m = re.search(r"<plan>(.*?)</plan>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if not m:
      return ""

  plan_text = m.group(1).strip()

  # Keep one action per non-empty line
  lines = [ln.strip() for ln in plan_text.splitlines() if ln.strip()]
  return "\n".join(lines)

In [ ]:
# Cell 7: Helper - validate a plan with VAL (GT domain + GT problem)
def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  proc = subprocess.run(cmd, capture_output=True, text=True)

  out_text = (proc.stdout or "").strip()
  err_text = (proc.stderr or "").strip()
  merged = (out_text + ("\n" + err_text if err_text else "")).strip()

  val_ok = "Plan valid" in merged
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [ ]:
# Cell 8: Helper - run one problem end-to-end (generate plan + VAL), persist outputs
def process_one_problem(problem_id: str):
  prompt = build_prompt(problem_id)
  raw_text = call_qwen(prompt)
  plan_text = extract_plan(raw_text)

  cand_dir = OUTPUT_ROOT / problem_id / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  raw_path = cand_dir / "raw.txt"
  plan_path = cand_dir / "plan.txt"
  val_log_path = cand_dir / "validator.log"
  err_path = cand_dir / "planner_error.txt"

  raw_path.write_text(raw_text or "")

  if not plan_text:
      plan_path.write_text("")
      err_msg = "Empty or missing <plan>...</plan> output."
      err_path.write_text(err_msg)
      return {
          "problem": problem_id,
          "plan_found": False,
          "val_ok": False,
          "planner_error": err_msg,
      }

  plan_path.write_text(plan_text + ("" if plan_text.endswith("\n") else "\n"))

  gt_problem_path = DATA_DIR / "pddl" / f"{problem_id}.pddl"
  if not gt_problem_path.exists():
      err_msg = f"Missing ground-truth problem file: {gt_problem_path}"
      err_path.write_text(err_msg)
      return {
          "problem": problem_id,
          "plan_found": True,
          "val_ok": False,
          "planner_error": err_msg,
      }

  val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_path)
  val_log_path.write_text(val_result.get("val_output", "") or "")

  return {
      "problem": problem_id,
      "plan_found": True,
      "val_ok": bool(val_result["val_ok"]),
      "planner_error": "",
  }

In [ ]:
# Cell 9: Generate only (no VAL)
smoke_problem = "p200"

prompt = build_prompt(smoke_problem)
raw_text = call_qwen(prompt)
plan_text = extract_plan(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "plan.txt").write_text((plan_text + "\n") if plan_text else "")
(cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_plan_chars:", len(plan_text or ""))
print("saved:", cand_dir)

In [ ]:
# Cell 10: VAL only (no model call), concise output
import contextlib
import io

smoke_problem = "p200"
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

plan_path = cand_dir / "plan.txt"
if not plan_path.exists():
  raise FileNotFoundError(f"Missing plan file: {plan_path}")

plan_text = plan_path.read_text().strip()
if not plan_text:
  summary = {
      "problem": smoke_problem,
      "plan_found": False,
      "val_ok": False,
      "planner_error": "Empty plan.txt (no actions).",
  }
  (cand_dir / "planner_error.txt").write_text(summary["planner_error"])
  print("Produced files:")
  print("- planner_error.txt")
  print("VAL result: not run (empty plan)")
  print("Summary:")
  print(json.dumps(summary, indent=2))
else:
  gt_problem_path = DATA_DIR / "pddl" / f"{smoke_problem}.pddl"
  if not gt_problem_path.exists():
      raise FileNotFoundError(f"Missing ground-truth problem file for VAL: {gt_problem_path}")

  with contextlib.redirect_stdout(io.StringIO()):
      val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_path)

  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  summary = {
      "problem": smoke_problem,
      "plan_found": True,
      "val_ok": bool(val_result["val_ok"]),
      "planner_error": "",
  }

  print("Produced files:")
  print("- validator.log")
  print(f"VAL result: {'valid' if val_result.get('val_ok') else 'invalid'}")
  print("Summary:")
  print(json.dumps(summary, indent=2))

In [ ]:
# Cell 11: Batch generate plans only (no VAL)
from tqdm.auto import tqdm

# Edit this range/selection as needed
BATCH_PROBLEMS_GEN = [f"p{i:02d}" for i in range(1, 11)]  # example: p01..p10

gen_rows = []

for pid in tqdm(BATCH_PROBLEMS_GEN, desc="Generating plans"):
  prompt = build_prompt(pid)
  raw_text = call_qwen(prompt)
  plan_text = extract_plan(raw_text)

  cand_dir = OUTPUT_ROOT / pid / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  (cand_dir / "raw.txt").write_text(raw_text or "")
  (cand_dir / "plan.txt").write_text((plan_text + "\n") if plan_text else "")
  (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

  gen_rows.append(
      {
          "problem": pid,
          "plan_extracted": bool(plan_text),
          "plan_chars": len(plan_text or ""),
          "cand_dir": str(cand_dir),
      }
  )

print("Batch generation done.")
print("Problems:", BATCH_PROBLEMS_GEN[0], "->", BATCH_PROBLEMS_GEN[-1], f"({len(BATCH_PROBLEMS_GEN)})")
print("Plans extracted:", sum(1 for r in gen_rows if r["plan_extracted"]), "/", len(gen_rows))


In [ ]:
# Cell 12: Batch VAL only (no model call)
from tqdm.auto import tqdm
import contextlib
import io

# Independent handle (does not depend on Cell 11 variables)
BATCH_PROBLEMS_VAL = [f"p{i:02d}" for i in range(1, 11)]  # example: p01..p10

val_rows = []

for pid in tqdm(BATCH_PROBLEMS_VAL, desc="Running VAL"):
  cand_dir = OUTPUT_ROOT / pid / "cand_00"
  plan_path = cand_dir / "plan.txt"
  gt_problem_path = DATA_DIR / "pddl" / f"{pid}.pddl"

  row = {
      "problem": pid,
      "plan_found": False,
      "val_ok": False,
      "planner_error": "",
  }

  if not plan_path.exists():
      row["planner_error"] = f"Missing plan file: {plan_path}"
      (cand_dir / "planner_error.txt").write_text(row["planner_error"])
      val_rows.append(row)
      continue

  plan_text = plan_path.read_text().strip()
  if not plan_text:
      row["planner_error"] = "Empty plan.txt (no actions)."
      (cand_dir / "planner_error.txt").write_text(row["planner_error"])
      val_rows.append(row)
      continue

  row["plan_found"] = True

  if not gt_problem_path.exists():
      row["planner_error"] = f"Missing ground-truth problem file: {gt_problem_path}"
      (cand_dir / "planner_error.txt").write_text(row["planner_error"])
      val_rows.append(row)
      continue

  with contextlib.redirect_stdout(io.StringIO()):
      val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_path)

  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
  row["val_ok"] = bool(val_result["val_ok"])

  val_rows.append(row)

print("Batch VAL done.")
print("Problems:", BATCH_PROBLEMS_VAL[0], "->", BATCH_PROBLEMS_VAL[-1], f"({len(BATCH_PROBLEMS_VAL)})")
print("Plan found:", sum(int(r["plan_found"]) for r in val_rows), "/", len(val_rows))
print("VAL valid:", sum(int(r["val_ok"]) for r in val_rows), "/", len(val_rows))


In [ ]:
# Cell 13: In-notebook metrics summary (no persistence)
import pandas as pd

# Independent handle for any selected batch
SUMMARY_PROBLEMS = [f"p{i:02d}" for i in range(1, 11)]  # example: p01..p10

rows = []
for pid in SUMMARY_PROBLEMS:
  cand_dir = OUTPUT_ROOT / pid / "cand_00"

  raw_path = cand_dir / "raw.txt"
  plan_path = cand_dir / "plan.txt"
  val_log_path = cand_dir / "validator.log"
  err_path = cand_dir / "planner_error.txt"

  raw_text = raw_path.read_text() if raw_path.exists() else ""
  plan_text = plan_path.read_text() if plan_path.exists() else ""
  val_log = val_log_path.read_text() if val_log_path.exists() else ""
  planner_error = err_path.read_text().strip() if err_path.exists() else ""

  plan_found = bool(plan_text.strip())
  val_ran = bool(val_log.strip())
  val_ok = "Plan valid" in val_log

  if val_ok:
      status = "plan_valid"
  elif plan_found and val_ran:
      status = "plan_invalid"
  else:
      status = "planner_failed"

  rows.append(
      {
          "problem_id": pid,
          "status": status,
          "plan_found": plan_found,
          "val_ran": val_ran,
          "val_ok": val_ok,
          "planner_error": planner_error,
          "raw_chars": len(raw_text),
          "plan_chars": len(plan_text),
          "cand_dir": str(cand_dir),
      }
  )

details_df = pd.DataFrame(rows).sort_values("problem_id").reset_index(drop=True)

total = len(details_df)
plan_found_n = int(details_df["plan_found"].sum())
val_ok_n = int(details_df["val_ok"].sum())
planner_failed_n = int((details_df["status"] == "planner_failed").sum())
plan_invalid_n = int((details_df["status"] == "plan_invalid").sum())

summary_df = pd.DataFrame(
  [
      {
          "num_problems": total,
          "plan_found_count": plan_found_n,
          "plan_found_rate": plan_found_n / total if total else 0.0,
          "val_ok_count": val_ok_n,
          "val_ok_rate": val_ok_n / total if total else 0.0,
          "planner_failed_count": planner_failed_n,
          "plan_invalid_count": plan_invalid_n,
      }
  ]
)

print("Summary:")
display(summary_df)

print("Per-problem details:")
display(details_df[["problem_id", "status", "plan_found", "val_ran", "val_ok", "planner_error"]])

# Keep in memory for Cell 14
SUMMARY_DF = summary_df
DETAILS_DF = details_df
SUMMARY_PROBLEMS_USED = SUMMARY_PROBLEMS


In [ ]:
# Cell 14: Persist summary/details to CSV (run only when you want to save)
from datetime import datetime

# Optional label to distinguish runs
SAVE_LABEL = "qwen3_32b_llm_planner"

# Save path under llm_planner output root
METRIC_DIR = OUTPUT_ROOT / "_metrics"
METRIC_DIR.mkdir(parents=True, exist_ok=True)

if "SUMMARY_DF" not in globals() or "DETAILS_DF" not in globals():
  raise RuntimeError("Run Cell 13 first to populate SUMMARY_DF and DETAILS_DF.")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
problems_tag = (
  f"{SUMMARY_PROBLEMS_USED[0]}_to_{SUMMARY_PROBLEMS_USED[-1]}"
  if len(SUMMARY_PROBLEMS_USED) > 1
  else SUMMARY_PROBLEMS_USED[0]
)

summary_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_summary_{ts}.csv"
details_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_details_{ts}.csv"

SUMMARY_DF.to_csv(summary_out, index=False)
DETAILS_DF.to_csv(details_out, index=False)

print("Saved:")
print("-", summary_out)
print("-", details_out)


# Patcher version 1: missing word "block"


In [ ]:
# Cell 15: Patcher setup (independent section, no problem handle here)
from pathlib import Path
import re
import json
import subprocess

# Project root (run notebook from formalizer-scaling/src/planner_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Independent paths for patcher section
DATA_DIR = PROJECT_ROOT / "data"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"

MODEL_DIRNAME = "qwen3_32b"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "llm_planner" / MODEL_DIRNAME

VAL_PATH = Path.home() / "tools" / "VAL" / "build" / "linux64" / "Release" / "bin" / "Validate"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("VAL exists:", VAL_PATH.exists())


In [ ]:
# Cell 16: Helper - patch a single plan line by adding "block" before bare integers
def patch_plan_line(line: str) -> str:
  s = line.strip()
  if not s:
      return line

  # Keep non-action lines unchanged
  if not (s.startswith("(") and s.endswith(")")):
      return line

  inner = s[1:-1].strip()
  if not inner:
      return line

  parts = inner.split()
  action = parts[0]
  args = parts[1:]

  patched_args = []
  for tok in args:
      # If argument is exactly an integer token (e.g., "12"), convert to "block12"
      if re.fullmatch(r"\d+", tok):
          patched_args.append(f"block{tok}")
      else:
          patched_args.append(tok)

  patched = "(" + " ".join([action] + patched_args) + ")"
  return patched


In [ ]:
# Cell 17: Helper - patch full plan text (line-by-line, preserving order)
def patch_plan_text(plan_text: str) -> str:
  lines = plan_text.splitlines()
  patched_lines = [patch_plan_line(ln) for ln in lines]

  # Keep one trailing newline if non-empty output
  patched_text = "\n".join(patched_lines)
  if patched_text and not patched_text.endswith("\n"):
      patched_text += "\n"
  return patched_text


In [ ]:
# Cell 18: Helper - run VAL for augmented plan and return result
def run_val_on_augmented_plan(domain_pddl_path: Path, problem_pddl_path: Path, augmented_plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(augmented_plan_path)]
  proc = subprocess.run(cmd, capture_output=True, text=True)

  out_text = (proc.stdout or "").strip()
  err_text = (proc.stderr or "").strip()
  merged = (out_text + ("\n" + err_text if err_text else "")).strip()

  val_ok = "Plan valid" in merged
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }


In [ ]:
# Cell 19: Patch run (single-problem handle) -> writes augmented_plan.txt
patch_problem = "p08"  # edit as needed

cand_dir = OUTPUT_ROOT / patch_problem / "cand_00"
plan_path = cand_dir / "plan.txt"
aug_plan_path = cand_dir / "augmented_plan.txt"

if not plan_path.exists():
  raise FileNotFoundError(f"Missing source plan file: {plan_path}")

original_plan = plan_path.read_text()
if not original_plan.strip():
  raise ValueError(f"Empty source plan file: {plan_path}")

augmented_plan = patch_plan_text(original_plan)
aug_plan_path.write_text(augmented_plan)

summary = {
  "problem": patch_problem,
  "source_plan_path": str(plan_path),
  "augmented_plan_path": str(aug_plan_path),
  "source_chars": len(original_plan),
  "augmented_chars": len(augmented_plan),
  "source_lines": len([ln for ln in original_plan.splitlines() if ln.strip()]),
  "augmented_lines": len([ln for ln in augmented_plan.splitlines() if ln.strip()]),
}

print("Patch done:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 20: Run VAL on augmented plan (single-problem handle) -> writes augmented_validator.log
aug_problem = "p08"  # edit as needed

cand_dir = OUTPUT_ROOT / aug_problem / "cand_00"
aug_plan_path = cand_dir / "augmented_plan.txt"
gt_problem_path = DATA_DIR / "pddl" / f"{aug_problem}.pddl"
aug_val_log_path = cand_dir / "augmented_validator.log"
aug_error_path = cand_dir / "augmented_error.txt"

if not aug_plan_path.exists():
  raise FileNotFoundError(f"Missing augmented plan file: {aug_plan_path}")
if not gt_problem_path.exists():
  raise FileNotFoundError(f"Missing ground-truth problem file: {gt_problem_path}")

aug_plan_text = aug_plan_path.read_text().strip()
if not aug_plan_text:
  err_msg = "Empty augmented_plan.txt (no actions)."
  aug_error_path.write_text(err_msg)
  print("Produced files:")
  print("- augmented_error.txt")
  print("VAL result: not run (empty augmented plan)")
  print("Summary:")
  print(json.dumps({"problem": aug_problem, "val_ok": False, "error": err_msg}, indent=2))
else:
  val_result = run_val_on_augmented_plan(DOMAIN_PDDL_PATH, gt_problem_path, aug_plan_path)
  aug_val_log_path.write_text(val_result.get("val_output", "") or "")

  summary = {
      "problem": aug_problem,
      "val_ok": bool(val_result["val_ok"]),
      "augmented_plan_path": str(aug_plan_path),
      "augmented_validator_log": str(aug_val_log_path),
  }

  print("Produced files:")
  print("- augmented_validator.log")
  print(f"VAL result: {'valid' if val_result['val_ok'] else 'invalid'}")
  print("Summary:")
  print(json.dumps(summary, indent=2))


# Patcher version 2: extra blank space between "block" and numbers


In [ ]:
# Cell 21: Patcher-2 setup (independent section, fixes "block 14" -> "block14")
from pathlib import Path
import re
import json
import subprocess

# Project root (run notebook from formalizer-scaling/src/planner_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Independent paths for patcher-2
DATA_DIR = PROJECT_ROOT / "data"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"

MODEL_DIRNAME = "qwen3_32b"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "llm_planner" / MODEL_DIRNAME

VAL_PATH = Path.home() / "tools" / "VAL" / "build" / "linux64" / "Release" / "bin" / "Validate"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("VAL exists:", VAL_PATH.exists())


In [ ]:
# Cell 22: Helper - patch a single plan line by merging "block <num>" into "block<num>"
def patch2_plan_line(line: str) -> str:
  s = line.strip()
  if not s:
      return line

  # Keep non-action lines unchanged
  if not (s.startswith("(") and s.endswith(")")):
      return line

  inner = s[1:-1].strip()
  if not inner:
      return line

  parts = inner.split()
  out = []
  i = 0
  while i < len(parts):
      tok = parts[i]
      if tok == "block" and i + 1 < len(parts) and re.fullmatch(r"\d+", parts[i + 1]):
          out.append(f"block{parts[i + 1]}")
          i += 2
      else:
          out.append(tok)
          i += 1

  return "(" + " ".join(out) + ")"


In [ ]:
# Cell 23: Helper - patch full plan text (line-by-line, preserving order)
def patch2_plan_text(plan_text: str) -> str:
  lines = plan_text.splitlines()
  patched_lines = [patch2_plan_line(ln) for ln in lines]

  patched_text = "\n".join(patched_lines)
  if patched_text and not patched_text.endswith("\n"):
      patched_text += "\n"
  return patched_text


In [ ]:
# Cell 24: Helper - run VAL for patcher-2 augmented plan and return result
def run_val_on_augmented2_plan(domain_pddl_path: Path, problem_pddl_path: Path, augmented_plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(augmented_plan_path)]
  proc = subprocess.run(cmd, capture_output=True, text=True)

  out_text = (proc.stdout or "").strip()
  err_text = (proc.stderr or "").strip()
  merged = (out_text + ("\n" + err_text if err_text else "")).strip()

  val_ok = "Plan valid" in merged
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }


In [ ]:
# Cell 25: Patcher-2 run (single-problem handle) -> writes augmented_plan.txt
patch2_problem = "p147"  # edit as needed

cand_dir = OUTPUT_ROOT / patch2_problem / "cand_00"
plan_path = cand_dir / "plan.txt"
aug_plan_path = cand_dir / "augmented_plan.txt"

if not plan_path.exists():
  raise FileNotFoundError(f"Missing source plan file: {plan_path}")

original_plan = plan_path.read_text()
if not original_plan.strip():
  raise ValueError(f"Empty source plan file: {plan_path}")

augmented_plan = patch2_plan_text(original_plan)
aug_plan_path.write_text(augmented_plan)

summary = {
  "problem": patch2_problem,
  "source_plan_path": str(plan_path),
  "augmented_plan_path": str(aug_plan_path),
  "source_chars": len(original_plan),
  "augmented_chars": len(augmented_plan),
  "source_lines": len([ln for ln in original_plan.splitlines() if ln.strip()]),
  "augmented_lines": len([ln for ln in augmented_plan.splitlines() if ln.strip()]),
}

print("Patch-2 done:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 26: Run VAL on patcher-2 augmented plan (single-problem handle) -> writes augmented_validator.log
aug2_problem = "p147"  # edit as needed

cand_dir = OUTPUT_ROOT / aug2_problem / "cand_00"
aug_plan_path = cand_dir / "augmented_plan.txt"
gt_problem_path = DATA_DIR / "pddl" / f"{aug2_problem}.pddl"
aug_val_log_path = cand_dir / "augmented_validator.log"
aug_error_path = cand_dir / "augmented_error.txt"

if not aug_plan_path.exists():
  raise FileNotFoundError(f"Missing augmented plan file: {aug_plan_path}")
if not gt_problem_path.exists():
  raise FileNotFoundError(f"Missing ground-truth problem file: {gt_problem_path}")

aug_plan_text = aug_plan_path.read_text().strip()
if not aug_plan_text:
  err_msg = "Empty augmented_plan.txt (no actions)."
  aug_error_path.write_text(err_msg)
  print("Produced files:")
  print("- augmented_error.txt")
  print("VAL result: not run (empty augmented plan)")
  print("Summary:")
  print(json.dumps({"problem": aug2_problem, "val_ok": False, "error": err_msg}, indent=2))
else:
  val_result = run_val_on_augmented2_plan(DOMAIN_PDDL_PATH, gt_problem_path, aug_plan_path)
  aug_val_log_path.write_text(val_result.get("val_output", "") or "")

  summary = {
      "problem": aug2_problem,
      "val_ok": bool(val_result["val_ok"]),
      "augmented_plan_path": str(aug_plan_path),
      "augmented_validator_log": str(aug_val_log_path),
  }

  print("Produced files:")
  print("- augmented_validator.log")
  print(f"VAL result: {'valid' if val_result['val_ok'] else 'invalid'}")
  print("Summary:")
  print(json.dumps(summary, indent=2))
